# 01 Extract — 小说原文 → 多轮对话数据

TaleTalk 流水线第一阶段：把一本小说的 txt 抽成「角色 + 台词」原始 jsonl，再切成多轮 ShareGPT SFT 数据，供 `02_train.ipynb` 训练 LoRA 使用。

**这一步只依赖 OpenAI 兼容的 LLM API**，可以是：

- 云端 API：DeepSeek / OpenAI / SiliconFlow / Moonshot / 阶跃 / 任意 OpenAI 兼容服务（填 .env 即可）
- 本地 vLLM / LLaMA Factory 起的 OpenAI server（同样走 .env 的 CUSTOM_*）

抽取支持断点续跑：相同 chunk 已抽过会从本地 cache 直接复用，重跑不会重复花钱。

## 0. 参数区

改完这个 cell 后**从头按顺序跑**即可。每次抽不同小说/不同角色就改这里。

In [ ]:
from pathlib import Path

# ====== 输入 ======
NOVEL_TXT = Path('/network-workspace/novels/十日终焉.txt')  # 小说原文 txt 路径
TARGET_ROLE = '齐夏'                                          # 想复刻的角色名（必须和小说里出现的称呼一致）
NOVEL_TITLE = '十日终焉'                                       # 仅用于 system prompt 文案
RUN_NAME = 'shiri_qixia'                                       # 数据集前缀，决定输出文件名

# ====== LLM 抽取后端 ======
# 在仓库根目录复制 .env.example 为 .env，按需填一个平台即可。
# 这里写默认平台名，notebook 跑起来会从 .env 读 base_url/key/model。
#   - deepseek / openai / siliconflow / moonshot / custom
#   - custom 适合本地 vLLM、LLaMA Factory api server、阶跃等任何 OpenAI 兼容端点
LLM_PLATFORM = 'custom'

# ====== 抽取行为 ======
MAX_WORKERS = 6              # 并发线程数。本地 vLLM 通常可以开大；公有云 API 看限流
CHUNK_SIZE_TOKENS = 1500     # 单 chunk 喂给 LLM 的 token 数。模型上下文小就调小

# ====== 多轮 SFT 转换 ======
VALID_RATIO = 0.05           # 验证集比例
MAX_CONVERSATIONS = 0        # 0 = 不截断；调试时填 100 之类先看效果
SEED = 42

# ====== 输出目录（这一步只产数据，不动模型）======
REPO_DIR = Path.cwd().resolve()
while REPO_DIR != REPO_DIR.parent and not (REPO_DIR / 'extract').is_dir():
    REPO_DIR = REPO_DIR.parent
assert (REPO_DIR / 'extract').is_dir(), f'未找到 taletalk 仓库根目录: {REPO_DIR}'

RAW_DIR = REPO_DIR / 'data' / 'raw'
SFT_DIR = REPO_DIR / 'data'
CACHE_DIR = REPO_DIR / 'cache'
for d in [RAW_DIR, SFT_DIR, CACHE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

RAW_JSONL = RAW_DIR / f'{RUN_NAME}_dialogues.jsonl'
TRAIN_JSON = SFT_DIR / f'{RUN_NAME}_chat_train.json'
VALID_JSON = SFT_DIR / f'{RUN_NAME}_chat_valid.json'

print('repo:', REPO_DIR)
print('novel:', NOVEL_TXT, 'exists=', NOVEL_TXT.exists())
print('target role:', TARGET_ROLE)
print('run name:', RUN_NAME)
print('platform:', LLM_PLATFORM)
print('raw jsonl:', RAW_JSONL)
print('train:', TRAIN_JSON)
print('valid:', VALID_JSON)

## 1. 安装抽取依赖

只装抽取阶段需要的几个小包，不会污染训练镜像。

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(REPO_DIR / 'requirements-extract.txt')])
print('extract deps installed')

## 2. 准备 .env

在仓库根目录手动复制 `.env.example` 为 `.env` 并填好 API 配置。下面这格只校验关键变量是否存在，不会打印 key。

In [ ]:
import os
from dotenv import load_dotenv

ENV_PATH = REPO_DIR / '.env'
if not ENV_PATH.exists():
    raise FileNotFoundError(f'请先 cp {REPO_DIR}/.env.example {ENV_PATH} 并填好 API 配置')
load_dotenv(ENV_PATH, override=True)

PLATFORM_VARS = {
    'deepseek': ('DEEPSEEK_API', 'DEEPSEEK_BASE_URL'),
    'openai': ('OPENAI_API_KEY', 'OPENAI_BASE_URL'),
    'siliconflow': ('SILICONFLOW_API_KEY', 'SILICONFLOW_BASE_URL'),
    'moonshot': ('MOONSHOT_API_KEY', 'MOONSHOT_BASE_URL'),
    'custom': ('CUSTOM_API_KEY', 'CUSTOM_BASE_URL'),
}
key_var, url_var = PLATFORM_VARS[LLM_PLATFORM]
key = os.environ.get(key_var, "").strip()
url = os.environ.get(url_var, "").strip()
assert key, f'{key_var} 未设置 (在 .env 里填写)'
assert url, f'{url_var} 未设置 (在 .env 里填写)'
print(f'platform={LLM_PLATFORM}  base_url={url}  key=***{key[-4:] if len(key)>4 else ""}')
# 让后续 Config 读到正确的平台
os.environ['LLM_PLATFORM'] = LLM_PLATFORM

## 3. 抽取原始多轮对话

调用 `extract` 模块（基于 KMnO4-zx/extract-dialogue 改造）。

- **断点续跑**：每个 chunk 抽完会写本地 cache，重跑只补做没完成的 chunk。
- **输出**：一行一条对话 jsonl，`{chunk_id, dialogue_index, role, dialogue}`。
- 抽全本一般 30 分钟 - 2 小时，看小说长度和 LLM 速度。

In [ ]:
import sys
sys.path.insert(0, str(REPO_DIR))

from extract import DialogueExtractor, Config

# 让 extract.Config 用 notebook 设定的目录
Config.CACHE_DIR = str(CACHE_DIR)
Config.CHUNK_SIZE = CHUNK_SIZE_TOKENS
Config.MAX_WORKERS = MAX_WORKERS
Config.set_platform(LLM_PLATFORM)

extractor = DialogueExtractor(
    platform=LLM_PLATFORM,
    max_workers=MAX_WORKERS,
    include_chunk_id=True,
    save_chunk_text=False,
)

assert NOVEL_TXT.exists(), f"小说原文不存在: {NOVEL_TXT}"
print(f"开始抽取: {NOVEL_TXT}")
print(f"输出: {RAW_JSONL}")
extractor.extract_dialogues_concurrent(
    file_path=str(NOVEL_TXT),
    output_file=str(RAW_JSONL),
)
print("抽取完成")

## 4. 速览抽取结果

In [ ]:
import json
from collections import Counter

lines = []
with RAW_JSONL.open(encoding='utf-8') as f:
    for ln in f:
        ln = ln.strip()
        if ln:
            lines.append(json.loads(ln))

print('总对话行数:', len(lines))
role_counts = Counter(l['role'] for l in lines)
print('Top 15 角色发言数:')
for role, n in role_counts.most_common(15):
    mark = '  <-- 目标' if role == TARGET_ROLE else ''
    print(f'  {role}: {n}{mark}')

target_n = role_counts.get(TARGET_ROLE, 0)
if target_n == 0:
    print(f'⚠️ 目标角色 {TARGET_ROLE!r} 一句都没抽到，检查名字是否拼写正确，或者小说里角色称呼是否一致')
elif target_n < 100:
    print(f'⚠️ 目标角色发言只有 {target_n} 条，数据量偏少，建议换更大角色或者更长小说')
else:
    print(f'✓ 目标角色发言 {target_n} 条')

## 5. 转成多轮 ShareGPT SFT 数据

保留每个 chunk 的真实对话流，目标角色每段连续发言作为 `gpt` turn，其他角色作为前置 `human` turn。这样模型学到的就是「跨轮维持人设」，而不是单轮 Q→A 切片。

In [ ]:
import subprocess
cmd = [
    'python3', str(REPO_DIR / 'scripts' / 'build_chat_sft_multiturn.py'),
    '--input', str(RAW_JSONL),
    '--target-role', TARGET_ROLE,
    '--run-name', RUN_NAME,
    '--novel-title', NOVEL_TITLE,
    '--out-dir', str(SFT_DIR),
    '--valid-ratio', str(VALID_RATIO),
    '--seed', str(SEED),
]
if MAX_CONVERSATIONS > 0:
    cmd += ['--max-conversations', str(MAX_CONVERSATIONS)]
print(' '.join(cmd))
subprocess.check_call(cmd)

## 6. 写 dataset_info.json

让 LLaMA Factory 知道训练数据集的列名/格式。`02_train.ipynb` 会读这个文件。

In [ ]:
import json

info_path = SFT_DIR / 'dataset_info.json'
info = {}
if info_path.exists():
    info = json.loads(info_path.read_text(encoding='utf-8'))

for split in ["train", "valid"]:
    key = f"{RUN_NAME}_{split}"
    info[key] = {
        "file_name": f"{RUN_NAME}_chat_{split}.json",
        "formatting": "sharegpt",
        "columns": {"messages": "conversations", "system": "system"},
        "tags": {"role_tag": "from", "content_tag": "value",
                 "user_tag": "human", "assistant_tag": "gpt"},
    }
info_path.write_text(json.dumps(info, ensure_ascii=False, indent=2), encoding="utf-8")
print('updated', info_path)
print('keys:', list(info.keys()))

## 7. 完成

现在 `data/{RUN_NAME}_chat_train.json` 已经就绪。

**下一步**：打开 `02_train.ipynb`，在它的参数区把 `RUN_NAME` 改成和本文件一致，跑训练。